In [ ]:
# Vamos calcular o erro do nosso modelo nos dados de teste (validação)

mae = mean_absolute_error(df_avaliado['Taxa de Congestionamento_mes (%)'], df_avaliado['Taxa_Prevista'])
rmse = np.sqrt(mean_squared_error(df_avaliado['Taxa de Congestionamento_mes (%)'], df_avaliado['Taxa_Prevista']))

print("=== Performance Global do Modelo (Baseline: Regressão Linear) ===")
print(f"Erro Absoluto Médio (MAE): {mae:.2f} p.p.")
print(f"Raiz do Erro Quadrático Médio (RMSE): {rmse:.2f} p.p.")

# Análise de Erro: Onde o modelo errou mais?
df_avaliado['Erro_Absoluto'] = abs(df_avaliado['Taxa de Congestionamento_mes (%)'] - df_avaliado['Taxa_Prevista'])
top_erros = df_avaliado.sort_values(by='Erro_Absoluto', ascending=False).head(5)

print("\n=== Top 5 Maiores Erros de Previsão ===")
display(top_erros[['comarca', 'serventia', 'mes_ref', 'Taxa de Congestionamento_mes (%)', 'Taxa_Prevista', 'Erro_Absoluto']])

In [ ]:
# Escolher uma serventia aleatória para plotar: Passado, Validação e Futuro

# Selecionar uma serventia que tenha dados suficientes
amostra = df_treino['serventia'].value_counts().index[0]
comarca_amostra = df_treino[df_treino['serventia'] == amostra]['comarca'].iloc[0]

# Filtrar dados
hist = df_treino[(df_treino['serventia'] == amostra) & (df_treino['comarca'] == comarca_amostra)]
val = df_avaliado[(df_avaliado['serventia'] == amostra) & (df_avaliado['comarca'] == comarca_amostra)]
fut = df_futuro[(df_futuro['serventia'] == amostra) & (df_futuro['comarca'] == comarca_amostra)]

plt.figure(figsize=(14, 6))

# Plotar Histórico (Treino)
plt.plot(hist['mes_ref'], hist['Taxa de Congestionamento_mes (%)'], label='Histórico (Real)', marker='o', color='blue')

# Plotar Validação (Real vs Previsto)
plt.plot(val['mes_ref'], val['Taxa de Congestionamento_mes (%)'], label='Validação (Real)', marker='x', color='green', linestyle='--')
plt.plot(val['mes_ref'], val['Taxa_Prevista'], label='Validação (Modelo)', marker='o', color='orange')

# Plotar Futuro
plt.plot(fut['data_futura'], fut['taxa_prevista'], label='Previsão Futura (3 meses)', marker='*', color='red', markersize=10)

plt.title(f'Projeção da Taxa de Congestionamento: {amostra} ({comarca_amostra})')
plt.ylabel('Taxa de Congestionamento (%)')
plt.xlabel('Data')
plt.legend()
plt.ylim(-5, 105) # Margem para visualização
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, max_error, r2_score

y = df_avaliado['Taxa de Congestionamento_mes (%)'].to_numpy()
yhat = df_avaliado['Taxa_Prevista'].to_numpy()
abs_err = np.abs(y - yhat)

# ponderação por volume (se existir)
w = df_avaliado['volume_total'].to_numpy() if 'volume_total' in df_avaliado.columns else None

resumo = {
    "MAE (p.p.)": mean_absolute_error(y, yhat),
    "RMSE (p.p.)": np.sqrt(mean_squared_error(y, yhat)),
    "Mediana erro abs (p.p.)": median_absolute_error(y, yhat),
    "Erro máximo (p.p.)": max_error(y, yhat),
    "R²": r2_score(y, yhat),
    "% erro ≤ 5 p.p.": (abs_err <= 5).mean() * 100,
    "% erro ≤ 10 p.p.": (abs_err <= 10).mean() * 100,
}

if w is not None:
    resumo["MAE ponderado (p.p.)"] = np.average(abs_err, weights=w)
    resumo["RMSE ponderado (p.p.)"] = np.sqrt(np.average((y - yhat)**2, weights=w))

pd.DataFrame([resumo]).round(2)
